# Cortical TRF analysis

This notebook checks how well cortical EEG responses can be predicted
from the speech envelope (gammatone predictors), and compares the
foreground story, the background story, and their mixture across the
different listening conditions (clean / diotic / binaural / dichotic).

Run this after data/bids_extraction.py and predictors/gammatone_predictors.py,
with analysis/experiment.py in the same folder (it is imported below).

## What happens, top to bottom

- Import eelbrain and load `e`, the pipeline object set up in
  `experiment.py` (this is where the actual EEG/predictor data gets
  pulled in).
- Mark bad channels and fit ICA for every subject (interactive, only
  needs doing once per subject - see the cell right after this).
- Load every subject's trigger/event information.
- Set up the display names, colors, and shared TRF-fitting settings
  reused by every section below.
- Define a region of interest (a set of electrodes over the front and
  middle of the head) and plot it, since later steps average over it
  instead of showing every electrode separately.
- Sanity check: confirm the basic sound-envelope predictor explains
  the EEG response at all, for every subject.
- Compare the foreground, background, and mixture predictors during
  the dichotic condition, split by which ear the background story
  came from, and test whether that difference is statistically real.
- Repeat that same foreground/background/mixture comparison across all
  three two-speaker conditions (diotic, binaural, dichotic), and test
  whether the differences between conditions are statistically real.
- Plot the TRFs themselves (the shape of the brain's response over
  time): first for a single speaker with no competing story, then for
  two speakers under each spatial condition.
- Find the timing of each response's peak and compare it by condition
  and by predictor.

Each section builds on the ones before it, so the cells are meant to
be run top to bottom, in order.

In [1]:
from pathlib import Path

from eelbrain import *
from experiment import e

# Where to save plots.
DST = Path('~/Desktop').expanduser()

INFO    :  *** BinauralCocktail initialized with root /Users/joshuaighalo/Github Repositories/dataset/cocktail/bids on 2026-09-06 00:19:33 ***
INFO    :  Using eelbrain 0.0.0, mne 1.12.1.


## Preprocessing (run once per subject)

Marks bad channels and fits ICA for every subject (`sub-01` through
`sub-13`). The actual bad-channel and ICA-component picks are
interactive - a plot opens and pauses the notebook until you close
it, then the next subject's plot opens automatically.

Results are cached per subject, so once a subject is done, re-running
this cell later never redoes it - safe to leave in and just run
through every time you run this notebook.

In [3]:
e.preprocess_all_subjects()


=== 01 ===
INFO    :  Raw 0.5-20: filtering for /Users/joshuaighalo/Github Repositories/dataset/cocktail/bids/sub-01/eeg/sub-01_task-cocktail_eeg.bdf...
Starting GUI. Quit the Python application to return to the shell...
Starting GUI. Quit the Python application to return to the shell...


KeyboardInterrupt: 

In [ ]:
e.load_events()

In [ ]:
EPOCHS = ['diotic', 'binaural', 'dichotic']

# Human-readable names and colors for the three predictors we compare
# throughout this notebook: the foreground story, the background story,
# and the mixture of both.
LABELS = {
    'gammatone_on_1': 'Foreground',
    'bg_gammatone_on_1': 'Background',
    'mix_gammatone_on_1': 'Mixture',
}
COLORS = {
    'gammatone_on_1': 'red',
    'bg_gammatone_on_1': 'blue',
    'mix_gammatone_on_1': '.3',
}

In [ ]:
# Settings shared by every TRF estimated in this notebook: which
# preprocessed EEG to use, how many subjects, the response time window
# to fit (-100 to 600 ms relative to sound onset). The boosting
# algorithm's own settings (error, basis, partitions,
# selective_stopping) live in experiment.py's `estimators` now, not
# here. See https://eelbrain.readthedocs.io/en/stable/experiment.html
# for what each of these controls.
PARAMETERS = {
    'raw': 'ica',
    'group': 'all',
    'samplingrate': 128,
    'data': 'eeg',
    'tstart': -0.100,
    'tstop': 0.600,
}

A small helper used throughout this notebook: eelbrain's own
`Pipeline` class has `load_model_test()` built in (runs a model
comparison and returns a statistical test result), but not a
plotting convenience on top of it (the old `trftools` package had
one, `show_model_test()`, which isn't available here - see the
Setup section of the README). This wraps `load_model_test()` for a
whole dictionary of named comparisons at once and plots a topomap of
each, passing the result straight to `plot.Topomap()` (the standard
way to show a masked-by-significance effect map in eelbrain). This
one is a reconstruction rather than a direct copy of the original
convenience function - if the plot doesn't look like what "masked by
significance" should look like, check `plot.Topomap`'s docs for the
exact way it expects a test result.

In [ ]:
def show_model_test(comparisons, vmax=None, **kwargs):
    for label, comparison in comparisons.items():
        result = e.load_model_test(comparison, **kwargs)
        display(plot.Topomap(result, vmax=vmax, title=label))

## Define a region of interest (ROI)

Cortical speech responses are usually strongest over fronto-central
electrodes, so later plots average over this set of sensors instead of
showing every electrode separately.

In [ ]:
data = e.load_trfs(1, 'gammatone-1', epoch='clean', **PARAMETERS)
eeg = data['ev']

ROI = [
    'AF3', 'F1', 'F3', 'F5',
    'FC5', 'FC3', 'FC1',
    'C1', 'C3', 'C5',
    'AF4', 'AFz',
    'Fz', 'F2', 'F4', 'F6',
    'FC6', 'FC4', 'FC2', 'FCz',
    'Cz', 'C2', 'C4', 'C6',
]

p = plot.SensorMap(eeg, mark=ROI)

In [ ]:
# A cleaner version of the same plot, for use in a figure.
p = plot.SensorMap(eeg, w=1, h=1, labels=False, mark=ROI)
p.save(DST / 'ROI.pdf')

## Envelope model

First, a sanity check: does the simple envelope predictor
("gammatone-1", the foreground story's overall loudness over time)
predict the EEG at all, for every subject?

In [ ]:
data = e.load_trfs('all', 'gammatone-1', epoch='clean', **PARAMETERS)
TOPO_ARGS = dict(vmax=0.005, clip='circle')
p = plot.Topomap('ev', data=data, **TOPO_ARGS)
p = plot.Topomap('ev', 'subject', rows=1, data=data, **TOPO_ARGS)

In [ ]:
# Same check, now also including the background story as a predictor,
# separately for each listening condition.
for epoch in ['diotic', 'dichotic', 'binaural']:
    data = e.load_trfs('all', 'gammatone-1 + bg~gammatone-1', epoch=epoch, **PARAMETERS)
    p = plot.Topomap('ev', data=data, title=epoch, **TOPO_ARGS)
    p = plot.Topomap('ev', 'subject', rows=1, data=data, **TOPO_ARGS)

## Dichotic: effect of ear

The auditory pathway crosses sides on its way to the brain, so a sound
in one ear is represented more strongly on the opposite side of the
brain. Since in the dichotic condition the background story always
comes from one ear, we might expect the cortical representation of
that background to depend on which ear it came from.

In [ ]:
# Compare three predictors (foreground, background, mixture) against a
# combined model that includes all three, to see how much unique
# predictive power each one adds.
FULL = "gammatone-on-1 + bg~gammatone-on-1 + mix~gammatone-on-1"
COMPARISONS = {
    'fg': f"{FULL} @ gammatone-on-1",
    'bg': f"{FULL} @ bg~gammatone-on-1",
    'mix': f"{FULL} @ mix~gammatone-on-1",
}

In [ ]:
show_model_test(COMPARISONS, **PARAMETERS, pmin=0.05, metric='ev', epoch='dichotic-left', vmax=.0005)

In [ ]:
show_model_test(COMPARISONS, **PARAMETERS, pmin=0.05, metric='ev', epoch='dichotic-right', vmax=.0005)

Collect the model comparisons for both ears so we can test whether the
difference is statistically meaningful.

In [ ]:
dss = []
for stream, comparison in COMPARISONS.items():
    for epoch in ['dichotic-left', 'dichotic-right']:
        data, result = e.load_model_test(comparison, **PARAMETERS, pmin=0.05, metric='ev', epoch=epoch, return_data=True)
        data = table.difference('ev', 'model', 'test', 'baseline', 'subject', data=data)
        data['stream', :] = stream
        dss.append(data)
data = combine(dss)
data['roi_ev'] = data['ev'].mean(sensor=ROI)

In [ ]:
test.ANOVA('roi_ev', 'stream * epoch * subject', sub="epoch != 'dichotic'", data=data)

In [ ]:
p = plot.Barplot('roi_ev', 'epoch', sub="stream == 'fg'", match='subject', data=data, h=2, w=2, corr=False, title='FG')
p.set_xtick_rotation(45)

## Effect of binaural cues

Now compare all three listening conditions that include a background
story (diotic, binaural, dichotic) to see whether the strength of the
background's cortical representation depends on the spatial cues
available to separate the two speakers.

In [ ]:
FULL = "gammatone-on-1 + bg~gammatone-on-1 + mix~gammatone-on-1"
COMPARISONS = {
    'fg': f"{FULL} @ gammatone-on-1",
    'bg': f"{FULL} @ bg~gammatone-on-1",
    'mix': f"{FULL} @ mix~gammatone-on-1",
}

Inadvertent attention switches (accidentally attending the wrong
story) are a possible confound, so check each condition individually
before comparing them.

In [ ]:
show_model_test(COMPARISONS, **PARAMETERS, pmin=0.05, metric='ev', epoch='diotic', vmax=.0005)

In [ ]:
show_model_test(COMPARISONS, **PARAMETERS, pmin=0.05, metric='ev', epoch='binaural', vmax=.0005)

In [ ]:
show_model_test(COMPARISONS, **PARAMETERS, pmin=0.05, metric='ev', epoch='dichotic', vmax=.0005)

### Is the difference between conditions reliable?

In [ ]:
dss = []
for stream, comparison in COMPARISONS.items():
    for epoch in ['diotic', 'dichotic', 'binaural']:
        data, result = e.load_model_test(comparison, **PARAMETERS, pmin=0.05, metric='ev', epoch=epoch, return_data=True)
        data = table.difference('ev', 'model', 'test', 'baseline', 'subject', data=data)
        data['stream', :] = stream
        dss.append(data)
data = combine(dss)
data['roi_ev'] = data['ev'].mean(sensor=ROI)
test.ANOVA('roi_ev', 'stream * epoch * subject', data=data)

In [ ]:
for stream in COMPARISONS:
    display(test.ANOVA('roi_ev', 'epoch * subject', sub=f"stream == '{stream}'", data=data, title=stream))
    p = plot.Barplot('roi_ev', 'epoch', sub=f"stream == '{stream}'", match='subject', data=data, h=2, w=2, corr=False, title=stream)
    p.set_xtick_rotation(45)

In [ ]:
# The p-value for the one comparison that came out significant above.
test.pairwise('roi_ev', 'epoch', sub="stream == 'bg'", match='subject', data=data, corr=False, title=False)

In [ ]:
# All pairwise comparisons, for reference.
test.pairwise('roi_ev', 'stream % epoch', match='subject', data=data, title=False)

## TRFs (temporal response functions)

A TRF shows the shape of the brain's response over time to a unit of
the predictor - essentially "if this sound feature occurred at time
0, how does the EEG respond over the following half second".

### Single speaker (no competing story)

In [ ]:
data = e.load_trfs(-1, 'gammatone-on-1', epoch='clean', **PARAMETERS)
trf = data['gammatone_on_1'].mean(sensor=ROI)
PLOT_ARGS = dict(frame='t', h=2.5, w=4, xlim=(-0.050, 0.550), clip=True, top=3.5, bottom=-0.5)
p = plot.UTSStat(trf * 1e3, **PLOT_ARGS)
p.save(DST / 'clean.pdf')

### Two speakers (foreground, background, and mixture)

In [ ]:
PLOT_ARGS['top'] = 2.5
all_dss = []
legend = True
for epoch in EPOCHS:
    data = e.load_trfs(-1, FULL, epoch=epoch, **PARAMETERS)

    dss = []
    for x in data.info['xs']:
        if '_on' not in x:
            # Only keep the onset-response predictors, matching FULL above.
            continue
        ds = data['subject',]
        ds[:, 'x'] = x
        ds[:, 'epoch'] = epoch
        ds['trf'] = data[x].mean(sensor=ROI)
        dss.append(ds)
        all_dss.append(ds)
    roi_data = combine(dss)
    p = plot.UTSStat('trf*1e3', 'x', data=roi_data, title=epoch.capitalize(), **PLOT_ARGS, labels=LABELS, colors=COLORS, legend=legend)
    p.save(DST / f'{epoch}.pdf')
    legend = False

roi_data = combine(all_dss)
# Smooth the TRFs to 512 samples for cleaner peak-time estimates below.
roi_data['trf_sm'] = resample(roi_data['trf'], 512)

In [ ]:
for epoch in EPOCHS:
    p = plot.UTSStat('trf_sm*1e3', 'x', sub=f"epoch == '{epoch}'", data=roi_data, title=epoch.capitalize(), **PLOT_ARGS, labels=LABELS, colors=COLORS, legend=legend)

## Peak response time

Find, for each subject/condition/predictor, the time of the largest
response between 20 and 130 ms - a typical window for the early
cortical response to speech onsets.

In [ ]:
roi_data['peak'] = roi_data['trf_sm'].sub(time=(0.02, 0.130)).argmax('time')

### Peak time by condition

In [ ]:
for epoch in EPOCHS:
    p = plot.Barplot('peak', 'x', match='subject', sub=f"epoch == '{epoch}'", data=roi_data, h=2, w=2.5, labels=LABELS, title=epoch, corr=False)
    display(test.pairwise('peak', 'x', match='subject', sub=f"epoch == '{epoch}'", data=roi_data, corr=False, title=False))
    p.set_xtick_rotation(30)

### Peak time by predictor (foreground / background / mixture)

In [ ]:
for x in COLORS:
    p = plot.Barplot('peak', 'epoch', match='subject', sub=f"x == '{x}'", data=roi_data, h=2, w=2.5, title=LABELS[x], corr=False)
    display(test.pairwise('peak', 'epoch', match='subject', sub=f"x == '{x}'", data=roi_data, corr=False, title=False))
    p.set_xtick_rotation(30)